# 02 — Split 80/20 por suerte y Pipeline de preprocesamiento (regresión)

**Objetivo:** aplicar las decisiones de imputación y outliers aprobadas en el EDA (notebook 01), sin
fuga de información:

1. Filtros de filas con reglas fijas (semilla y error del denominador), **antes** del split.
2. Hold-out 80/20 con `GroupShuffleSplit` por suerte (`Hacienda + Suerte`): ninguna suerte queda a la vez en train y en test.
3. `Pipeline` = `ReglasNegocio` (sin estado) + `ColumnTransformer` (medianas, categorías poco frecuentes y escalado,
   con `fit` solo sobre `X_train`, o sobre el train de cada fold en la CV).

El código reutilizable está en `src/datos.py` y `src/preprocesamiento.py`. Este notebook solo **verifica**;
el benchmark y los modelos van en el notebook siguiente.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
from src.datos import (CV_GRUPOS, LLUVIAS_VENTANA, cargar_historico, filtrar_filas, grupos,
                       preparar_xy, split_regresion)
from src.preprocesamiento import (LLUVIA_2M, LLUVIA_FC, LLUVIAS_TRIMESTRALES, ReglasNegocio,
                                  construir_preprocesador)

# Categorías de test que no aparecen en train → van a la categoría "infrequent" (comportamiento buscado)
warnings.filterwarnings("ignore", message="Found unknown categories")
pd.set_option("display.width", 140)

## 1. Filtros de filas (reglas fijas, antes del split)

In [2]:
df_sin_semilla = cargar_historico()
df = filtrar_filas(df_sin_semilla)
print(f"Sin semilla: {len(df_sin_semilla):,} | tras quitar TCH extremo en ≤ 1 ha: {len(df):,} "
      f"(−{len(df_sin_semilla) - len(df)})")
assert len(df_sin_semilla) == 20_733 and len(df_sin_semilla) - len(df) == 29

Sin semilla: 20,733 | tras quitar TCH extremo en ≤ 1 ha: 20,704 (−29)


## 2. Split 80/20 por suerte

`Suerte` sola no identifica un lote: el mismo código se repite en varias haciendas. La clave de grupo es
`Hacienda + Suerte`.

In [3]:
print(f"Códigos de Suerte: {df['Suerte'].nunique():,} | grupos Hacienda+Suerte: {grupos(df).nunique():,} "
      f"| registros por grupo: {len(df) / grupos(df).nunique():.1f}")
print("Códigos de Suerte presentes en más de una hacienda:", (df.groupby("Suerte")["Hacienda"].nunique() > 1).sum())

train, test = split_regresion(df)
g_train, g_test = set(grupos(train)), set(grupos(test))
frac_test = len(test) / len(df)
print(f"\nTrain: {len(train):,} filas, {len(g_train):,} suertes | Test: {len(test):,} filas, {len(g_test):,} suertes "
      f"| % filas en test: {frac_test:.1%}")
assert not g_train & g_test, "hay suertes en train y en test"
assert 0.19 <= frac_test <= 0.21

Códigos de Suerte: 1,202 | grupos Hacienda+Suerte: 5,760 | registros por grupo: 3.6
Códigos de Suerte presentes en más de una hacienda: 360

Train: 16,599 filas, 4,608 suertes | Test: 4,105 filas, 1,152 suertes | % filas en test: 19.8%


Las distribuciones de los targets en train y test deberían ser parecidas (el KS es solo informativo).

In [4]:
filas = []
for t in ["TCH", "%Sac.Caña"]:
    a, b = train[t].dropna(), test[t].dropna()
    filas.append({"target": t, **{f"train_{k}": v for k, v in a.describe()[["mean", "25%", "50%", "75%"]].items()},
                  **{f"test_{k}": v for k, v in b.describe()[["mean", "25%", "50%", "75%"]].items()},
                  "KS_p": ks_2samp(a, b).pvalue})
pd.DataFrame(filas).set_index("target").round(3)

,train_mean,train_25%,train_50%,train_75%,test_mean,test_25%,test_50%,test_75%,KS_p
target,,,,,,,,,
TCH,130.223,109.000,130.000,151.000,129.773,110.623,129.404,149.00,0.063
%Sac.Caña,12.317,11.607,12.364,13.087,12.337,11.666,12.401,13.09,0.200


## 3. `ReglasNegocio`: conteo de cada regla

Es un paso sin estado (no aprende nada en `fit`), así que se puede aplicar sobre las 20 733 filas sin semilla
solo para contar. Los conteos deben coincidir con `01_eda_regresion.ipynb` §8.

In [5]:
cols_entrada = list(preparar_xy(df_sin_semilla, "TCH")[0].columns) + ["Sac.Caña Precosecha"]
antes = df_sin_semilla[cols_entrada]
despues = ReglasNegocio().fit_transform(antes)

edad = antes["Edad Ult Cos"]
cambio = lambda c: (antes[c] != despues[c]) & antes[c].gt(0) & despues[c].notna()
conteos = {
    "Semanas mad. → NaN (fecha imposible)": (antes["Semanas mad."].notna() & despues["Semanas mad."].isna()).sum(),
    "Dosis → NaN (producto con dosis 0)": (despues["Dosis Madurante"].isna()).sum(),
    "Producto = DESCONOCIDO (dosis sin nombre)": (despues["Producto"] == "DESCONOCIDO").sum(),
    "Filas con ventana trimestral recortada a 900": pd.concat([cambio(c) for c in LLUVIAS_TRIMESTRALES], axis=1).any(axis=1).sum(),
    "2 Meses Ant. recortada a 600": (cambio(LLUVIA_2M) & (despues[LLUVIA_2M] == 600)).sum(),
    "9-FC recortada a 300 mm/mes": (cambio(LLUVIA_FC) & (edad > 9)).sum(),
    "9-FC > 0 con Edad ≤ 9 → 0 (ventana inexistente)": ((antes[LLUVIA_FC] > 0) & (edad <= 9)).sum(),
    "DDUlt.Riego → NaN (sin riego registrado)": despues["DDUlt.Riego"].isna().sum(),
    "Edad Ult Cos recortada a 36 meses (tope estadístico)": (antes["Edad Ult Cos"] != despues["Edad Ult Cos"]).sum(),
    "Precosecha > 25 → NaN (error de captura)": (antes["Sac.Caña Precosecha"].gt(25) & despues["Sac.Caña Precosecha"].isna()).sum(),
}
esperado = {"Semanas mad. → NaN (fecha imposible)": 10, "Dosis → NaN (producto con dosis 0)": 24,
            "Producto = DESCONOCIDO (dosis sin nombre)": 3, "Filas con ventana trimestral recortada a 900": 28,
            "2 Meses Ant. recortada a 600": 13, "9-FC recortada a 300 mm/mes": 6,
            "9-FC > 0 con Edad ≤ 9 → 0 (ventana inexistente)": 23, "DDUlt.Riego → NaN (sin riego registrado)": 12_523,
            "Edad Ult Cos recortada a 36 meses (tope estadístico)": 1, "Precosecha > 25 → NaN (error de captura)": 10}
tabla = pd.DataFrame({"filas": conteos, "esperado": esperado})
display(tabla)
assert (tabla["filas"] == tabla["esperado"]).all()

,filas,esperado
Semanas mad. → NaN (fecha imposible),10,10
Dosis → NaN (producto con dosis 0),24,24
Producto = DESCONOCIDO (dosis sin nombre),3,3
Filas con ventana trimestral recortada a 900,28,28
2 Meses Ant. recortada a 600,13,13
9-FC recortada a 300 mm/mes,6,6
9-FC > 0 con Edad ≤ 9 → 0 (ventana inexistente),23,23
DDUlt.Riego → NaN (sin riego registrado),12523,12523
Edad Ult Cos recortada a 36 meses (tope estadístico),1,1
Precosecha > 25 → NaN (error de captura),10,10


In [6]:
nombres_prod = despues.loc[~despues["Producto"].isin(["NO APLICADO", "DESCONOCIDO"]), "Producto"].nunique()
print(f"Productos con nombre: {antes['Producto'].nunique()} → {nombres_prod} (se unifican grafías con/sin acento)"
      " + categorías NO APLICADO y DESCONOCIDO")
print("% sin dato por ventana de lluvia (0 → NaN):")
print(despues[LLUVIAS_VENTANA].isna().mean().round(3).to_string())
print("\n9-FC con Edad ≤ 9 (debe ser 0, no NaN):", despues.loc[edad <= 9, LLUVIA_FC].value_counts(dropna=False).to_dict())
print("Máximos tras recorte:", despues[LLUVIAS_VENTANA].max().to_dict())

Productos con nombre: 18 → 17 (se unifican grafías con/sin acento) + categorías NO APLICADO y DESCONOCIDO
% sin dato por ventana de lluvia (0 → NaN):
Lluvias (2 Meses Ant.)    0.485
Lluvias 0 -3              0.448
Lluvias tres a seis       0.448
Lluvias seis a nueve      0.452
Luvias 9 -FC              0.450

9-FC con Edad ≤ 9 (debe ser 0, no NaN): {0.0: 234}
Máximos tras recorte: {'Lluvias (2 Meses Ant.)': 600.0, 'Lluvias 0 -3': 900.0, 'Lluvias tres a seis': 900.0, 'Lluvias seis a nueve': 900.0, 'Luvias 9 -FC': 2752.0}


## 4. Preprocesador: `fit` en train, `transform` en test

Tres variantes de entrada (`preparar_xy`): TCH, %Sac.Caña (sin las 156 filas sin target) y %Sac.Caña con
`Sac.Caña Precosecha` (solo 2017–2021 con dato real). Las tres salen de las **mismas** particiones train/test.

In [7]:
variantes = {"TCH": ("TCH", False), "%Sac": ("%Sac.Caña", False), "%Sac + precosecha": ("%Sac.Caña", True)}
datos = {}
filas = []
for nombre, (target, con_pre) in variantes.items():
    X_tr, y_tr, g_tr = preparar_xy(train, target, con_pre)
    X_te, y_te, _ = preparar_xy(test, target, con_pre)
    datos[nombre] = (X_tr, y_tr, g_tr, X_te, y_te)
    for tipo in ["lineal", "arbol"]:
        pre = construir_preprocesador(tipo, con_pre).fit(X_tr)  # fit SOLO sobre train
        Z_tr, Z_te = pre.transform(X_tr), pre.transform(X_te)
        assert Z_tr.isna().sum().sum() == 0 and Z_te.isna().sum().sum() == 0
        assert list(Z_tr.columns) == list(Z_te.columns)
        filas.append({"variante": nombre, "tipo": tipo, "n_train": len(Z_tr), "n_test": len(Z_te),
                      "n_features": Z_tr.shape[1]})
pd.DataFrame(filas)

,variante,tipo,n_train,n_test,n_features
0,TCH,lineal,16599,4105,82
1,TCH,arbol,16599,4105,85
2,%Sac,lineal,16472,4076,82
3,%Sac,arbol,16472,4076,85
4,%Sac + precosecha,lineal,7317,1794,81
5,%Sac + precosecha,arbol,7317,1794,85


In [8]:
pre_lineal = construir_preprocesador("lineal").fit(datos["TCH"][0])
nombres = pre_lineal.get_feature_names_out()
print(len(nombres), "features (lineal, TCH):")
print(list(nombres))

82 features (lineal, TCH):
['Lluvias (2 Meses Ant.)', 'Lluvias 0 -3', 'Lluvias tres a seis', 'Lluvias seis a nueve', 'Luvias 9 -FC', 'missingindicator_Lluvias (2 Meses Ant.)', 'missingindicator_Lluvias 0 -3', 'missingindicator_Lluvias tres a seis', 'missingindicator_Lluvias seis a nueve', 'missingindicator_Luvias 9 -FC', 'Dosis Madurante', 'Semanas mad._sp_0', 'Semanas mad._sp_1', 'Semanas mad._sp_2', 'Semanas mad._sp_3', 'Semanas mad._sp_4', 'Semanas mad._sp_5', 'Semanas mad._sp_6', 'DDUlt.Riego', 'Edad Ult Cos', 'Cod.Estado #', 'Dist Km', 'anio', 'M3 Riego', 'Variedad_CC01-1940', 'Variedad_CC05-430', 'Variedad_CC11-600', 'Variedad_CC85-92', 'Variedad_CC93-4181', 'Variedad_CC93-4418', 'Variedad_CC97-7170', 'Variedad_RB73-2223', 'Variedad_V71-51', 'Variedad_VARIAS', 'Variedad_infrequent_sklearn', 'Suelo_BALLESTEROS', 'Suelo_BURRIGA', 'Suelo_CERRITO', 'Suelo_CERRITO-NIMA', 'Suelo_CORINTIAS', 'Suelo_Desconocido', 'Suelo_GALPON', 'Suelo_GENOVA', 'Suelo_GUADUAL', 'Suelo_JORDAN', 'Suelo_JUA

## 5. Anti-leakage: las medianas salen solo de train

Se comparan las medianas que aprendió el imputador de lluvias con las calculadas a mano sobre `X_train`.
Después se reajusta sobre el train de un fold de `GroupKFold` y se comprueba que las medianas cambian
(cada fold recalcula sus estadísticos).

In [9]:
X_tr, y_tr, g_tr = datos["TCH"][:3]

def imputador_lluvias(pre):
    return pre.named_steps["columnas"].named_transformers_["lluvias"].steps[0][1]

imp = imputador_lluvias(pre_lineal)
manual = ReglasNegocio().fit_transform(X_tr).groupby(["Zona", "mes_cosecha"])[LLUVIAS_VENTANA].median()
pd.testing.assert_frame_equal(imp.medianas_[0], manual)
print("Medianas Zona × mes del imputador == medianas manuales sobre X_train ✔")

idx_fold, _ = next(CV_GRUPOS.split(X_tr, y_tr, groups=g_tr))
imp_fold = imputador_lluvias(construir_preprocesador("lineal").fit(X_tr.iloc[idx_fold]))
dif = (imp_fold.medianas_[0] - imp.medianas_[0]).abs()
print(f"Fold 1: {(dif > 0).to_numpy().sum()} de {dif.notna().to_numpy().sum()} medianas Zona × mes distintas a las de train completo")
assert (dif > 0).to_numpy().any()

Medianas Zona × mes del imputador == medianas manuales sobre X_train ✔


Fold 1: 240 de 300 medianas Zona × mes distintas a las de train completo


## 6. Smoke test: el Pipeline completo corre dentro de la CV por grupos

`LinearRegression` con `GroupKFold(5)` sobre train. Solo comprueba que el pipeline funcione de punta a punta
(el preprocesador se reajusta en cada fold). **No** es el benchmark: los resultados se reportan en el
notebook de modelos. Un R² > 0.9 sería alarma de leakage.

In [10]:
filas = []
for nombre, (X_tr, y_tr, g_tr, _, _) in datos.items():
    modelo = Pipeline([("pre", construir_preprocesador("lineal", nombre.endswith("precosecha"))),
                       ("modelo", LinearRegression())])
    cv = cross_validate(modelo, X_tr, y_tr, groups=g_tr, cv=CV_GRUPOS,
                        scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"])
    filas.append({"variante": nombre, "R2_cv": cv["test_r2"].mean(),
                  "RMSE_cv": -cv["test_neg_root_mean_squared_error"].mean(),
                  "MAE_cv": -cv["test_neg_mean_absolute_error"].mean()})
smoke = pd.DataFrame(filas).set_index("variante").round(3)
display(smoke)
assert (smoke["R2_cv"] < 0.9).all(), "R² sospechoso: revisar leakage"

,R2_cv,RMSE_cv,MAE_cv
variante,,,
TCH,0.328,26.588,20.415
%Sac,0.387,0.898,0.691
%Sac + precosecha,0.417,0.776,0.602


## Conclusión

- **Split:** 16 599 filas / 4 608 suertes en train, 4 105 / 1 152 en test (19.8 %), sin suertes compartidas.
  Los targets tienen distribuciones equivalentes en las dos particiones.
- **Reglas fijas:** cada conteo coincide con el EDA (§8).
- **Preprocesador:** 0 NaN después de `transform` en train y test, con las mismas columnas. Las medianas salen solo
  de train y se recalculan en cada fold.
- **Smoke test:** el pipeline corre dentro de `GroupKFold(5)`. Los R² (~0.3–0.4) no muestran señales de
  leakage. Son solo de control: el benchmark formal va en el notebook siguiente.